# Logistic Regression: запуск экспериментов

001–004 сравнивают preprocessing и веса, 005 выбирает основу по tuned OOF, 006 добавляет 60/90 s к 005. Каждый подход: C=1, затем поиск C на тех же трёх folds, затем два refit на train и оценка val. Выбор основы и C не использует val.

**SMOKE=True** включает проверочный режим: девять людей train, два val, короткие участки вокруг стадий; это проверка работоспособности, не качество полного эксперимента. Для Kaggle задайте пути и SMOKE=False. Обучение вызывается через uv в закреплённом окружении, независимо от Python текущего ядра. Запускайте конфигурации последовательно.

In [ ]:
import sys
from pathlib import Path

# На Kaggle задайте абсолютный путь к каталогу кода проекта.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if (
    not (PROJECT_ROOT / "pyproject.toml").is_file()
    or not (PROJECT_ROOT / "src" / "__init__.py").is_file()
):
    raise FileNotFoundError("Укажите PROJECT_ROOT: каталог с pyproject.toml и src/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "raw"  # На Kaggle укажите реальный Input root.
OUTPUT_ROOT = PROJECT_ROOT / "artefacts"  # Путь должен быть доступен для записи.
SMOKE = False
CONFIGS = [PROJECT_ROOT / "configs" / f"{i:03d}_linear.yaml" for i in range(1, 7)]
command = [
    "uv",
    "run",
    "--locked",
    "python",
    "-m",
    "src.run",
    *map(str, CONFIGS),
    "--data-root",
    str(DATA_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
]
if SMOKE:
    command.append("--smoke")
print(command)

Первый запуск извлекает признаки. Кэш 30 s общий для всех подходов; длинные окна нужны только 006. Подбор C не пересчитывает признаки. Повторный запуск возобновляет завершённые trials при идентичных настройках и fingerprint кода. Изменённый эксперимент не перезаписывается.

Границы найденного C допускают одну дополнительную декаду; все trials сохраняются. OOF после подбора оптимистичен и не является независимой оценкой. Не меняйте выбор конфигурации по val в ходе этого запуска.

In [ ]:
from src.notebook import run_command

run_command(command, cwd=PROJECT_ROOT)

Результаты — в OUTPUT_ROOT/<ID>/, smoke — в OUTPUT_ROOT/smoke/<ID>/. Сравнение результатов находится в 03_eval. В experiments.csv записываются только полноценные завершённые запуски. При смене кода/зависимостей используйте отдельный output root для новых smoke-проверок.